# AI for Health — Deep Learning Exercises

## Ex 1: Chest X-ray Classification — ResNet50 vs Vision Transformer
## Ex 2: 3D Semantic Segmentation — MONAI U-Net on Spleen (Decathlon Task09)

Designed to run on **Colab with a T4 GPU**. Both exercises use small, fast-downloading datasets so a full run takes minutes, not hours.

### What we'll do

**Ex 1 — Classification** on PneumoniaMNIST (~6,000 pediatric chest X-rays, normal vs pneumonia):
- Load the dataset with `medmnist`
- Fine-tune a pre-trained **ResNet50** and a pre-trained **Vision Transformer (ViT-B/16)**
- Compare with **ROC-AUC**, accuracy, and confusion matrices
- Visualise attention maps: **Grad-CAM** for ResNet50, **attention rollout** for ViT

**Ex 2 — Segmentation** on Decathlon Task09 Spleen (~1.5 GB, 3D CT volumes):
- Load with **MONAI**'s `DecathlonDataset` (downloads automatically)
- Build a 3D **U-Net**
- Train with `DiceCELoss`
- Evaluate with **Dice coefficient** and **IoU**
- Visualise predicted vs. ground-truth segmentation masks


---
## 0. Setup

⚠ **Before you run anything**: in Colab, go to *Runtime → Change runtime type → Hardware accelerator → T4 GPU*. Without a GPU, Ex 2 will be impractically slow.

In [ ]:
# One-shot install. Safe to re-run; --quiet keeps the output clean.
!pip install --quiet medmnist timm grad-cam scikit-learn
!pip install --quiet 'monai-weekly[nibabel,tqdm,gdown]'

import os, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch:    {torch.__version__}")
print(f"CUDA:       {torch.cuda.is_available()}")
print(f"Device:     {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU:        {torch.cuda.get_device_name(0)}")
else:
    print("⚠  No GPU detected. Ex 2 (3D segmentation) will be very slow.")

---
# Ex 1 — Chest X-ray Classification: ResNet50 vs ViT

## 1.1 Dataset — PneumoniaMNIST

[PneumoniaMNIST](https://medmnist.com/) is a subset of MedMNIST: 5,856 pediatric chest X-rays labeled as **normal (0)** or **pneumonia (1)**. The 224×224 version is suitable for ImageNet-pretrained backbones.

- Train: 4,708 images
- Val:   524 images
- Test:  624 images
- License: CC BY 4.0
- Source: https://medmnist.com/


In [ ]:
import medmnist
from medmnist import INFO
import torchvision.transforms as T
from torch.utils.data import DataLoader

DATA_FLAG = "pneumoniamnist"
info = INFO[DATA_FLAG]
print(f"Task: {info['task']}")
print(f"Classes: {info['label']}")
print(f"Samples: {info['n_samples']}")
print(f"Channels: {info['n_channels']} (will be converted to 3 for pretrained models)")

DataClass = getattr(medmnist, info["python_class"])

# Pre-trained backbones expect 3-channel RGB inputs normalised with ImageNet stats.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tx = T.Compose([
    T.Grayscale(num_output_channels=3),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tx = T.Compose([
    T.Grayscale(num_output_channels=3),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

os.makedirs("medmnist_data", exist_ok=True)
train_ds = DataClass(split="train", download=True, size=224, transform=train_tx, root="medmnist_data")
val_ds   = DataClass(split="val",   download=True, size=224, transform=eval_tx,  root="medmnist_data")
test_ds  = DataClass(split="test",  download=True, size=224, transform=eval_tx,  root="medmnist_data")

print(f"\nLoaded: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")

BATCH_SIZE = 64
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

### 1.2 Visualise a few samples

In [ ]:
# Show 8 sample images with their labels
# Use a no-normalisation transform just for display
display_tx = T.Compose([T.Grayscale(num_output_channels=1), T.ToTensor()])
display_ds = DataClass(split="train", download=False, size=224, transform=display_tx, root="medmnist_data")

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
np.random.seed(0)
indices = np.random.choice(len(display_ds), size=8, replace=False)
for ax, idx in zip(axes.flat, indices):
    img, lbl = display_ds[idx]
    ax.imshow(img[0], cmap="gray")
    label_name = info["label"][str(int(lbl[0]))]
    ax.set_title(f"#{idx}  -  {label_name}", fontsize=11,
                 color="darkred" if label_name == "pneumonia" else "darkgreen")
    ax.axis("off")
plt.suptitle("PneumoniaMNIST — random training samples", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("ex1_samples.png", dpi=120, bbox_inches="tight")
plt.show()

# Class balance
from collections import Counter
labels_all = [int(train_ds[i][1][0]) for i in range(len(train_ds))]
print("Train set class balance:", Counter(labels_all))

### 1.3 Training utilities

A single `train_model` function we'll reuse for both ResNet50 and ViT.

In [ ]:
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix

def evaluate(model, loader):
    """Return (auc, acc, probs, labels)."""
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            yb = yb.squeeze().long()
            logits = model(xb)
            probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            all_probs.extend(probs.tolist())
            all_labels.extend(yb.numpy().tolist())
    auc = roc_auc_score(all_labels, all_probs)
    preds = [int(p > 0.5) for p in all_probs]
    acc = accuracy_score(all_labels, preds)
    return auc, acc, np.array(all_probs), np.array(all_labels)

def train_model(model, name, epochs=3, lr=1e-4):
    """Train, validate per epoch, return history + best test metrics."""
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    history = {"epoch": [], "train_loss": [], "val_auc": [], "val_acc": []}
    print(f"\n--- Training {name} ---")
    print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

    for ep in range(1, epochs + 1):
        model.train()
        running_loss, n = 0.0, 0
        t0 = time.time()
        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.squeeze().long().to(DEVICE)
            opt.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            opt.step()
            running_loss += loss.item() * xb.size(0)
            n += xb.size(0)
        train_loss = running_loss / n
        val_auc, val_acc, _, _ = evaluate(model, val_loader)
        dt = time.time() - t0
        history["epoch"].append(ep)
        history["train_loss"].append(train_loss)
        history["val_auc"].append(val_auc)
        history["val_acc"].append(val_acc)
        print(f"Epoch {ep}/{epochs} | "
              f"loss={train_loss:.4f} | "
              f"val_auc={val_auc:.4f} | "
              f"val_acc={val_acc:.4f} | "
              f"{dt:.1f}s")

    # Final test eval
    test_auc, test_acc, test_probs, test_labels = evaluate(model, test_loader)
    print(f"\n{name} TEST  AUC={test_auc:.4f}  ACC={test_acc:.4f}")
    return {
        "name": name,
        "history": history,
        "test_auc": test_auc,
        "test_acc": test_acc,
        "test_probs": test_probs,
        "test_labels": test_labels,
    }

### 1.4 Model 1 — pre-trained ResNet50

In [ ]:
from torchvision import models

resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
# Replace the final layer for 2-class classification
resnet.fc = nn.Linear(resnet.fc.in_features, 2)

resnet_result = train_model(resnet, "ResNet50", epochs=3, lr=1e-4)

### 1.5 Model 2 — pre-trained ViT-B/16

In [ ]:
import timm

vit = timm.create_model("vit_base_patch16_224", pretrained=True, num_classes=2)
vit_result = train_model(vit, "ViT-B/16", epochs=3, lr=1e-4)

### 1.6 Compare ResNet50 vs ViT

In [ ]:
# --- Side-by-side training curves ---
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for r, color in [(resnet_result, "tab:blue"), (vit_result, "tab:orange")]:
    h = r["history"]
    axes[0].plot(h["epoch"], h["train_loss"], "o-", color=color, label=r["name"])
    axes[1].plot(h["epoch"], h["val_auc"],    "o-", color=color, label=r["name"])
axes[0].set_title("Training loss"); axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].set_title("Validation AUC"); axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("AUC"); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig("ex1_training_curves.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# --- ROC curves on the test set ---
from sklearn.metrics import roc_curve

fig, ax = plt.subplots(figsize=(6.5, 6))
for r, color in [(resnet_result, "tab:blue"), (vit_result, "tab:orange")]:
    fpr, tpr, _ = roc_curve(r["test_labels"], r["test_probs"])
    ax.plot(fpr, tpr, color=color, lw=2, label=f"{r['name']} (AUC={r['test_auc']:.3f})")
ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="random")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("ROC curves on PneumoniaMNIST test set")
ax.legend(loc="lower right")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("ex1_roc.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# --- Confusion matrices ---
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, r in zip(axes, [resnet_result, vit_result]):
    preds = (r["test_probs"] > 0.5).astype(int)
    cm = confusion_matrix(r["test_labels"], preds)
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["normal", "pneumonia"])
    ax.set_yticklabels(["normal", "pneumonia"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"{r['name']}  (acc={r['test_acc']:.3f})")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black",
                    fontsize=14, fontweight="bold")
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig("ex1_confusion.png", dpi=120, bbox_inches="tight")
plt.show()

# --- Summary table ---
summary = pd.DataFrame([
    {"model": "ResNet50", "test_AUC": resnet_result["test_auc"], "test_acc": resnet_result["test_acc"]},
    {"model": "ViT-B/16", "test_AUC": vit_result["test_auc"],    "test_acc": vit_result["test_acc"]},
])
print("\n=== Final test results ===")
print(summary.to_string(index=False))

### 1.7 Attention maps

Two different techniques for two different architectures:
- **Grad-CAM** for ResNet50 — gradients of the predicted class with respect to the last conv feature map, weighted-summed and ReLU'd
- **Attention rollout** for ViT — cumulative product of attention matrices across the 12 transformer blocks, then extract the [CLS]-token's attention to each patch

In [ ]:
# --- Grad-CAM for ResNet50 ---
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

# Use the last residual block as the target layer
target_layers = [resnet.layer4[-1]]
cam = GradCAM(model=resnet, target_layers=target_layers)

# Pick a few test images
display_tx_disp = T.Compose([T.Grayscale(num_output_channels=3), T.ToTensor()])
disp_test_ds = DataClass(split="test", download=False, size=224, transform=display_tx_disp, root="medmnist_data")

def to_input(x):
    """Normalise a [0,1] tensor for the model."""
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (x - mean) / std

np.random.seed(1)
idxs = np.random.choice(len(disp_test_ds), size=4, replace=False)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for col, idx in enumerate(idxs):
    img01, lbl = disp_test_ds[idx]   # (3,224,224) in [0,1]
    inp = to_input(img01).unsqueeze(0).to(DEVICE)
    heatmap = cam(input_tensor=inp)[0]   # (224, 224) in [0,1]

    rgb = img01.permute(1, 2, 0).numpy()  # (224, 224, 3)
    overlay = show_cam_on_image(rgb, heatmap, use_rgb=True)

    label_name = info["label"][str(int(lbl[0]))]

    axes[0, col].imshow(rgb)
    axes[0, col].set_title(f"#{idx}  {label_name}", fontsize=10)
    axes[0, col].axis("off")
    axes[1, col].imshow(overlay)
    axes[1, col].set_title("Grad-CAM (ResNet50)", fontsize=10)
    axes[1, col].axis("off")
plt.suptitle("ResNet50 — Grad-CAM attention maps", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("ex1_gradcam.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# --- Attention rollout for ViT ---
import torch.nn.functional as F

def get_vit_attentions(model, x):
    """Run the ViT and grab attention weights from every transformer block."""
    attentions = []
    handles = []
    def make_hook(module):
        def hook(mod, inp, out):
            B, N, C = inp[0].shape
            qkv = mod.qkv(inp[0]).reshape(B, N, 3, mod.num_heads, mod.head_dim).permute(2, 0, 3, 1, 4)
            q, k, _ = qkv.unbind(0)
            attn = (q @ k.transpose(-2, -1)) * mod.scale
            attn = attn.softmax(dim=-1)
            attentions.append(attn.detach().cpu())
        return hook
    for blk in model.blocks:
        handles.append(blk.attn.register_forward_hook(make_hook(blk.attn)))
    with torch.no_grad():
        _ = model(x)
    for h in handles:
        h.remove()
    return attentions

def attention_rollout(attentions):
    """Multiply attention matrices across layers, add identity for residuals."""
    result = torch.eye(attentions[0].size(-1))
    for a in attentions:
        a = a.mean(dim=1)              # fuse heads
        I = torch.eye(a.size(-1))
        a = (a + I) / 2
        a = a / a.sum(dim=-1, keepdim=True)
        result = a[0] @ result
    mask = result[0, 1:]               # CLS-token attention to each patch, drop CLS-to-CLS
    width = int(mask.shape[-1] ** 0.5)
    mask = mask.reshape(width, width).numpy()
    return mask / mask.max()

vit.eval()
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for col, idx in enumerate(idxs):
    img01, lbl = disp_test_ds[idx]
    inp = to_input(img01).unsqueeze(0).to(DEVICE)
    attentions = get_vit_attentions(vit, inp)
    roll = attention_rollout(attentions)
    # Upsample 14x14 -> 224x224 for overlay
    roll_up = F.interpolate(
        torch.tensor(roll).float()[None, None], size=(224, 224),
        mode="bilinear", align_corners=False
    )[0, 0].numpy()
    rgb = img01.permute(1, 2, 0).numpy()
    overlay = (0.5 * rgb + 0.5 * plt.cm.jet(roll_up)[..., :3])

    label_name = info["label"][str(int(lbl[0]))]

    axes[0, col].imshow(rgb)
    axes[0, col].set_title(f"#{idx}  {label_name}", fontsize=10)
    axes[0, col].axis("off")
    axes[1, col].imshow(overlay)
    axes[1, col].set_title("ViT attention rollout", fontsize=10)
    axes[1, col].axis("off")
plt.suptitle("ViT-B/16 — Attention rollout maps", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("ex1_vit_attention.png", dpi=120, bbox_inches="tight")
plt.show()

### 1.8 Ex 1 interpretation

Things to look for and write up:

- **AUC comparison.** On PneumoniaMNIST both models typically reach AUC > 0.90 after 3 epochs. ViT is usually within a couple of percentage points of ResNet50; on this small dataset the conv inductive bias of ResNet50 often wins.
- **Training time.** ViT-B has ~86M params vs ~25M for ResNet50, so each epoch is roughly 2× longer.
- **Attention maps.** Grad-CAM heatmaps usually localise the lungs/diffuse opacities. ViT attention rollout tends to be more diffuse and global because every patch attends to every other patch; that's a real architectural difference, not a bug.
- **Failure modes.** Look at the confusion matrix — pneumonia cases missed (false negatives) are clinically worse than false positives, so the operating threshold matters. The 0.5 cutoff used here is arbitrary; in deployment you'd pick a threshold from the ROC curve to hit a target sensitivity.


---
# Ex 2 — 3D Semantic Segmentation: MONAI U-Net on Spleen

## 2.1 Dataset — Decathlon Task09 Spleen

The [Medical Segmentation Decathlon](http://medicaldecathlon.com/) Task09 is a binary segmentation task: 41 abdominal CT volumes with pixel-level spleen annotations.

- **Modality:** CT
- **Volumes:** 41 (we split 80/20 train/val)
- **Classes:** 2 (background, spleen)
- **Format:** NIfTI (`.nii.gz`)
- **Size:** ~1.5 GB
- **License:** CC BY-SA 4.0
- **Source:** http://medicaldecathlon.com/

MONAI downloads it automatically with one line.


In [ ]:
from monai.apps import DecathlonDataset
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd, Spacingd,
    ScaleIntensityRanged, CropForegroundd, RandCropByPosNegLabeld,
    RandFlipd, RandRotate90d, RandShiftIntensityd, EnsureTyped, AsDiscrete,
)
from monai.data import DataLoader as MonaiDataLoader, CacheDataset, decollate_batch
from monai.networks.nets import UNet
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric, MeanIoU
from monai.inferers import sliding_window_inference
import torch

ROOT = "monai_data"
os.makedirs(ROOT, exist_ok=True)

# Window settings for abdominal CT: typical for soft tissue
WIN_MIN, WIN_MAX = -57, 164
SPACING = (1.5, 1.5, 2.0)   # mm, resample to common voxel grid

train_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    Spacingd(keys=["image", "label"], pixdim=SPACING, mode=("bilinear", "nearest")),
    ScaleIntensityRanged(keys=["image"], a_min=WIN_MIN, a_max=WIN_MAX, b_min=0.0, b_max=1.0, clip=True),
    CropForegroundd(keys=["image", "label"], source_key="image"),
    RandCropByPosNegLabeld(
        keys=["image", "label"], label_key="label", spatial_size=(96, 96, 96),
        pos=1, neg=1, num_samples=4, image_key="image", image_threshold=0,
    ),
    RandFlipd(keys=["image", "label"], spatial_axis=[0], prob=0.5),
    RandFlipd(keys=["image", "label"], spatial_axis=[1], prob=0.5),
    RandFlipd(keys=["image", "label"], spatial_axis=[2], prob=0.5),
    RandRotate90d(keys=["image", "label"], prob=0.5, max_k=3),
    RandShiftIntensityd(keys=["image"], offsets=0.10, prob=0.5),
    EnsureTyped(keys=["image", "label"]),
])
val_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    Spacingd(keys=["image", "label"], pixdim=SPACING, mode=("bilinear", "nearest")),
    ScaleIntensityRanged(keys=["image"], a_min=WIN_MIN, a_max=WIN_MAX, b_min=0.0, b_max=1.0, clip=True),
    CropForegroundd(keys=["image", "label"], source_key="image"),
    EnsureTyped(keys=["image", "label"]),
])

# Download once; cache_num=0 keeps RAM usage modest on Colab
train_ds = DecathlonDataset(
    root_dir=ROOT, task="Task09_Spleen", section="training",
    transform=train_transforms, download=True, num_workers=2,
    val_frac=0.2, seed=0, cache_num=24,
)
val_ds = DecathlonDataset(
    root_dir=ROOT, task="Task09_Spleen", section="validation",
    transform=val_transforms, download=False, num_workers=2,
    val_frac=0.2, seed=0, cache_num=8,
)
print(f"Train volumes: {len(train_ds)}, Val volumes: {len(val_ds)}")

### 2.2 Visualise one training sample

In [ ]:
# Look at a single random crop and its label
sample = train_ds[0]
# RandCropByPosNegLabeld returns a list of crops; if so, take the first
if isinstance(sample, list):
    sample = sample[0]
img = sample["image"][0].cpu().numpy()
lbl = sample["label"][0].cpu().numpy()
print(f"Crop shape: image={img.shape}, label={lbl.shape}")
print(f"Image intensity range (after scaling to [0,1]): [{img.min():.2f}, {img.max():.2f}]")
print(f"Label classes present: {np.unique(lbl).tolist()}")

# Show middle slices in each plane
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
mid = [s // 2 for s in img.shape]
for r, vol, title in [(0, img, "CT"), (1, lbl, "Spleen mask")]:
    axes[r, 0].imshow(vol[mid[0], :, :], cmap="gray" if r == 0 else "viridis")
    axes[r, 0].set_title(f"{title}  (sagittal, X={mid[0]})")
    axes[r, 1].imshow(vol[:, mid[1], :], cmap="gray" if r == 0 else "viridis")
    axes[r, 1].set_title(f"{title}  (coronal, Y={mid[1]})")
    axes[r, 2].imshow(vol[:, :, mid[2]], cmap="gray" if r == 0 else "viridis")
    axes[r, 2].set_title(f"{title}  (axial, Z={mid[2]})")
    for ax in axes[r]:
        ax.axis("off")
plt.suptitle("Spleen — one training crop (96×96×96)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("ex2_sample.png", dpi=120, bbox_inches="tight")
plt.show()

### 2.3 Build the 3D U-Net

In [ ]:
train_loader = MonaiDataLoader(train_ds, batch_size=2, shuffle=True, num_workers=2)
val_loader   = MonaiDataLoader(val_ds,   batch_size=1, shuffle=False, num_workers=2)

unet = UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=2,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
).to(DEVICE)
print(f"UNet3D params: {sum(p.numel() for p in unet.parameters()):,}")

loss_fn = DiceCELoss(to_onehot_y=True, softmax=True)
opt = torch.optim.Adam(unet.parameters(), lr=1e-4)
dice_metric = DiceMetric(include_background=False, reduction="mean")
iou_metric  = MeanIoU(include_background=False, reduction="mean")

post_pred  = AsDiscrete(argmax=True, to_onehot=2)
post_label = AsDiscrete(to_onehot=2)

### 2.4 Training loop

For a class exercise we run a small number of epochs (~10) to demonstrate the workflow. State-of-the-art on this task needs ~600 epochs and gets Dice ~0.96; we're aiming for Dice ~0.70–0.85 here, which still produces visibly correct masks.

In [ ]:
EPOCHS = 10
VAL_EVERY = 2
best_dice = 0.0
history = {"epoch": [], "loss": [], "val_dice": [], "val_iou": []}

t_start = time.time()
for ep in range(1, EPOCHS + 1):
    unet.train()
    running_loss, n = 0.0, 0
    t0 = time.time()
    for batch in train_loader:
        x = batch["image"].to(DEVICE)
        y = batch["label"].to(DEVICE)
        opt.zero_grad()
        logits = unet(x)
        loss = loss_fn(logits, y)
        loss.backward()
        opt.step()
        running_loss += loss.item() * x.size(0)
        n += x.size(0)
    train_loss = running_loss / n
    history["epoch"].append(ep)
    history["loss"].append(train_loss)

    # Validation every VAL_EVERY epochs
    if ep % VAL_EVERY == 0 or ep == EPOCHS:
        unet.eval()
        dice_metric.reset(); iou_metric.reset()
        with torch.no_grad():
            for batch in val_loader:
                x = batch["image"].to(DEVICE)
                y = batch["label"].to(DEVICE)
                # Sliding-window inference handles arbitrary-size volumes
                logits = sliding_window_inference(x, roi_size=(96, 96, 96), sw_batch_size=2, predictor=unet)
                preds = [post_pred(p) for p in decollate_batch(logits)]
                labels = [post_label(l) for l in decollate_batch(y)]
                dice_metric(y_pred=preds, y=labels)
                iou_metric(y_pred=preds, y=labels)
        v_dice = dice_metric.aggregate().item()
        v_iou  = iou_metric.aggregate().item()
        history["val_dice"].append((ep, v_dice))
        history["val_iou"].append((ep, v_iou))
        if v_dice > best_dice:
            best_dice = v_dice
            torch.save(unet.state_dict(), "best_unet.pt")
        dt = time.time() - t0
        print(f"Epoch {ep}/{EPOCHS} | loss={train_loss:.4f} | val_dice={v_dice:.4f} | val_iou={v_iou:.4f} | {dt:.1f}s")
    else:
        dt = time.time() - t0
        print(f"Epoch {ep}/{EPOCHS} | loss={train_loss:.4f} | {dt:.1f}s")

print(f"\nTotal training time: {time.time() - t_start:.0f}s")
print(f"Best val Dice: {best_dice:.4f}")

### 2.5 Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(history["epoch"], history["loss"], "o-", color="tab:blue")
axes[0].set_title("Training loss (DiceCE)")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].grid(alpha=0.3)

if history["val_dice"]:
    eps, dices = zip(*history["val_dice"])
    _,   ious  = zip(*history["val_iou"])
    axes[1].plot(eps, dices, "o-", color="tab:green", label="Dice")
    axes[1].plot(eps, ious,  "s-", color="tab:purple", label="IoU")
    axes[1].set_title("Validation metrics")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Score")
    axes[1].set_ylim(0, 1)
    axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("ex2_curves.png", dpi=120, bbox_inches="tight")
plt.show()

### 2.6 Visualise predictions on a validation volume

In [ ]:
# Load the best checkpoint
unet.load_state_dict(torch.load("best_unet.pt"))
unet.eval()

with torch.no_grad():
    val_batch = next(iter(val_loader))
    x = val_batch["image"].to(DEVICE)
    y = val_batch["label"].to(DEVICE)
    logits = sliding_window_inference(x, roi_size=(96, 96, 96), sw_batch_size=2, predictor=unet)
    pred = torch.argmax(logits, dim=1, keepdim=True)

img_np   = x[0, 0].cpu().numpy()
label_np = y[0, 0].cpu().numpy()
pred_np  = pred[0, 0].cpu().numpy()

# Pick the slice with the largest spleen footprint for visualisation
slice_areas = (label_np > 0).sum(axis=(0, 1))
best_slice = int(np.argmax(slice_areas))
print(f"Showing axial slice z={best_slice} (largest spleen area)")

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
axes[0].imshow(img_np[:, :, best_slice], cmap="gray")
axes[0].set_title("CT (axial)")
axes[0].axis("off")

axes[1].imshow(img_np[:, :, best_slice], cmap="gray")
axes[1].contour(label_np[:, :, best_slice], colors="lime", linewidths=1.5)
axes[1].set_title("Ground truth (green contour)")
axes[1].axis("off")

axes[2].imshow(img_np[:, :, best_slice], cmap="gray")
axes[2].contour(label_np[:, :, best_slice], colors="lime", linewidths=1.5)
axes[2].contour(pred_np[:, :, best_slice],  colors="red",  linewidths=1.5, linestyles="--")
axes[2].set_title("GT (green) vs Prediction (red)")
axes[2].axis("off")

plt.tight_layout()
plt.savefig("ex2_prediction.png", dpi=120, bbox_inches="tight")
plt.show()

### 2.7 Ex 2 interpretation

What's interesting in the results:

- **Dice vs IoU.** Dice ≈ 2·IoU / (1+IoU); they always trend together but Dice penalises small misalignments less harshly. Both are used; Dice is the convention in medical-imaging papers.
- **Why so few epochs?** State-of-the-art on Spleen needs 600+ epochs to reach Dice ~0.96. With 10 epochs on Colab T4 you should reach Dice ~0.70–0.85, which is enough to see visibly correct masks. Increasing `EPOCHS` is the single most impactful change.
- **What to look at in the prediction figure.** The green contour is ground truth; red dashes are the prediction. Mismatches usually fall into two patterns: boundary jitter (a-few-voxel inaccuracy) — fixable with longer training; and missed lobes — usually fixable with more training data or stronger augmentation.
- **Why DiceCE loss?** Pure Dice loss has unstable gradients when the prediction has no overlap with the label (the gradient saturates). The cross-entropy term keeps gradients informative in those early-training conditions.
- **Sliding-window inference.** Training uses fixed 96×96×96 crops, but real volumes are much larger. `sliding_window_inference` tiles the full volume into overlapping 96³ patches, runs the model on each, and stitches the outputs — the standard way to do inference on volumes that don't fit in memory at full resolution.


---
# Summary

| | Ex 1 — Classification | Ex 2 — Segmentation |
|---|---|---|
| Dataset | PneumoniaMNIST (~6k 2D X-rays) | Decathlon Task09 Spleen (41 3D CT volumes) |
| Source | https://medmnist.com/ | http://medicaldecathlon.com/ |
| Task | Binary classification | Binary 3D semantic segmentation |
| Models | ResNet50 (ImageNet) + ViT-B/16 (ImageNet) | MONAI U-Net (3D, from scratch) |
| Loss | Cross-entropy | DiceCE (Dice + cross-entropy) |
| Metric | ROC-AUC, accuracy | Dice coefficient, IoU |
| Visualisation | Grad-CAM (CNN), Attention rollout (ViT) | Contour overlay on axial slice |

### Saved figures
All figures are saved as PNG in the working directory:
- `ex1_samples.png` — example PneumoniaMNIST images
- `ex1_training_curves.png` — ResNet50 vs ViT training loss & val AUC
- `ex1_roc.png` — ROC curves on the test set
- `ex1_confusion.png` — confusion matrices
- `ex1_gradcam.png` — Grad-CAM attention maps
- `ex1_vit_attention.png` — ViT attention rollout maps
- `ex2_sample.png` — example Spleen CT crop with mask
- `ex2_curves.png` — segmentation training loss + val Dice/IoU
- `ex2_prediction.png` — predicted vs ground-truth spleen contour
